# Stress index — Apple Health Auto Export analysis

Reusable pipeline for the **Health Auto Export** iOS app's CSV format: a daily-aggregate export
(`<SUBJECT>_HealthAutoExport-<start>-<end>.csv`) and a per-workout export
(`<SUBJECT>_Workouts-<start>_<time>-<end>_<time>.csv`).

(An earlier version of this notebook explored a Kaggle biometric dataset with different column
names/measures as an early prototype; this version replaces it to match the real export format.)

## Loading a new export

Drop the two CSVs for a new month (or a different person) straight into this folder and set
`SUBJECT` below to the filename prefix (or leave it — the loader picks the most recent matching
pair by filename). Nothing else needs to change unless the export's column set is different.

In [ ]:
import glob
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path(".")
SUBJECT = "AH"  # filename prefix -- change to match a different export

health_files = sorted(glob.glob(str(DATA_DIR / f"{SUBJECT}_HealthAutoExport-*.csv")))
workout_files = sorted(glob.glob(str(DATA_DIR / f"{SUBJECT}_Workouts-*.csv")))

HEALTH_CSV = health_files[-1] if health_files else None
WORKOUTS_CSV = workout_files[-1] if workout_files else None
print("Health export:  ", HEALTH_CSV)
print("Workouts export:", WORKOUTS_CSV)

In [ ]:
# Health Auto Export column -> short internal name. Only columns present in
# the file are kept, so a sparser or richer export still loads fine.
DAILY_COLUMN_MAP = {
    "Active Energy (kcal)": "active_kcal",
    "Apple Exercise Time (min)": "exercise_min",
    "Apple Stand Time (min)": "stand_min",
    "Heart Rate [Min] (count/min)": "hr_min",
    "Heart Rate [Max] (count/min)": "hr_max",
    "Heart Rate [Avg] (count/min)": "hr_avg",
    "Heart Rate Variability (ms)": "hrv",
    "Respiratory Rate (count/min)": "resp_rate",
    "Resting Heart Rate (count/min)": "rhr",
    "Sleep Analysis [Total] (hr)": "sleep_total",
    "Sleep Analysis [Deep] (hr)": "sleep_deep",
    "Sleep Analysis [REM] (hr)": "sleep_rem",
    "Sleep Analysis [Awake] (hr)": "sleep_awake",
    "Step Count (count)": "steps",
    "VO2 Max (ml/(kg·min))": "vo2max",
    "Weight (lb)": "weight_lb",
    "Blood Oxygen Saturation (%)": "spo2",
}

WORKOUT_COLUMN_MAP = {
    "Workout Type": "type",
    "Active Energy (kcal)": "active_kcal",
    "Avg. Heart Rate (count/min)": "avg_hr",
    "Max. Heart Rate (count/min)": "max_hr",
    "Distance (km)": "distance_km",
}


def load_daily_export(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["Date/Time"]).dt.normalize()
    df = df.set_index("date").sort_index()
    present = {k: v for k, v in DAILY_COLUMN_MAP.items() if k in df.columns}
    return df[list(present)].rename(columns=present)


def load_workouts(path):
    df = pd.read_csv(path)
    df["start"] = pd.to_datetime(df["Start"])
    df["end"] = pd.to_datetime(df["End"])
    df["date"] = df["start"].dt.normalize()
    df["duration_min"] = pd.to_timedelta(df["Duration"]).dt.total_seconds() / 60
    present = {k: v for k, v in WORKOUT_COLUMN_MAP.items() if k in df.columns}
    df = df.rename(columns=present)
    keep = ["type", "date", "start", "end", "duration_min", "active_kcal", "avg_hr", "max_hr", "distance_km"]
    return df[[c for c in keep if c in df.columns]].sort_values("start").reset_index(drop=True)


daily = load_daily_export(HEALTH_CSV)
workouts = load_workouts(WORKOUTS_CSV)
daily.tail()

In [ ]:
def drop_partial_last_day(df, steps_col="steps", threshold=0.25):
    """Health Auto Export's most recent day is often a partial export
    (whatever synced before the file was generated) -- drop it if its
    step count is far below the recent typical level so it doesn't skew
    averages/trends."""
    if len(df) < 2 or steps_col not in df:
        return df
    last, typical = df[steps_col].iloc[-1], df[steps_col].iloc[:-1].median()
    if pd.notna(last) and pd.notna(typical) and last < threshold * typical:
        print(f"Dropping {df.index[-1].date()} as a partial day (steps {last:.0f} vs typical {typical:.0f})")
        return df.iloc[:-1]
    return df


# daily_all keeps every full day in the export (not just the baseline window)
# so a day outside PERIOD_START-PERIOD_END -- like a specific event day -- can
# still be scored against the baseline later. See "Scoring a single day" below.
daily_all = drop_partial_last_day(daily)
stat_cols = [c for c in ["hrv", "rhr", "sleep_total", "resp_rate", "steps", "active_kcal", "exercise_min", "vo2max"] if c in daily_all]
daily_all[stat_cols].describe().round(2)

In [ ]:
# Sleep architecture, not just duration: deep-sleep share of total sleep
# (recovery depth) and time awake during the sleep session (fragmentation,
# already loaded as sleep_awake). Verified Total == Core + Deep + REM in
# this export, so Awake is already excluded from sleep_total -- these are
# independent signals, not a re-statement of it. Computed on daily_all so
# it's available both inside and outside the baseline window.
if "sleep_deep" in daily_all and "sleep_total" in daily_all:
    daily_all["sleep_deep_pct"] = daily_all["sleep_deep"] / daily_all["sleep_total"]

daily_all[[c for c in ["sleep_total", "sleep_deep_pct", "sleep_awake"] if c in daily_all]].describe().round(3)

In [ ]:
# Restrict to the analysis/baseline window. The ramp-in (missing sleep/resp
# on several days, sparse workouts before Jul 10) and the very first day
# (partial tracking) are excluded so this period is a clean reference.
PERIOD_START = "2026-07-05"
PERIOD_END = "2026-07-24"

daily_full = daily_all.loc[PERIOD_START:PERIOD_END]
workouts = workouts[(workouts["date"] >= PERIOD_START) & (workouts["date"] <= PERIOD_END)].reset_index(drop=True)

print(f"Analysis window: {PERIOD_START} to {PERIOD_END} ({len(daily_full)} days, {len(workouts)} workouts)")

In [ ]:
corr_cols = [c for c in ["hrv", "rhr", "resp_rate", "sleep_total", "exercise_min", "steps", "active_kcal"] if c in daily_full]
daily_full[corr_cols].corr(numeric_only=True).round(2)

### Trend vs. day-to-day noise

A metric can move over the window for two very different reasons: a **slow trend** (fitness
adapting, growth, a gradually worsening or easing situation) or **day-to-day noise/acute swings**
(that day's training, sleep, or stress). Averaging both into one correlation or one z-score blurs
them together -- e.g. RHR trended down over this window as fitness improved, while HRV stayed
essentially flat day to day, which is part of why the two didn't correlate. `trend_decompose` fits
`metric ~ day index` to separate the two, so a future period can tell "this metric moved because
of a slow trend" from "this metric moved acutely."

In [ ]:
def trend_decompose(series, threshold=0.3):
    """Fits `series ~ day index` on the non-null values to separate a slow
    within-window trend from day-to-day residual noise. `threshold` is the
    |r| above which the trend is flagged as worth calling out -- 0.3 is a
    conventional "weak-to-moderate" cutoff, not a statistical test; with
    only a few weeks of daily data, treat this as a prompt to look closer,
    not a verdict."""
    s = series.dropna()
    if len(s) < 5:
        return {"r_vs_day": None, "slope_per_day": None, "trending": False}
    x = np.arange(len(s))
    r = np.corrcoef(x, s.to_numpy())[0, 1]
    slope = np.polyfit(x, s.to_numpy(), 1)[0]
    return {"r_vs_day": round(float(r), 2), "slope_per_day": round(float(slope), 4), "trending": abs(r) >= threshold}


def trend_summary(df, cols, threshold=0.3):
    rows = {c: trend_decompose(df[c], threshold=threshold) for c in cols if c in df}
    return pd.DataFrame(rows).T


TREND_COLS = ["hrv", "rhr", "resp_rate", "sleep_total", "sleep_deep_pct", "sleep_awake"]
trend_tbl = trend_summary(daily_full, TREND_COLS)
trend_tbl

In [ ]:
for metric, row in trend_tbl.iterrows():
    if row["trending"]:
        direction = "declining" if row["slope_per_day"] < 0 else "rising"
        print(
            f"{metric}: {direction} ~{abs(row['slope_per_day']):.3f}/day across the window "
            f"(r={row['r_vs_day']:+.2f} vs. day index) -- day-to-day moves in this metric are "
            f"riding a slow trend, not purely acute."
        )
    else:
        print(f"{metric}: no meaningful window trend (r={row['r_vs_day']}) -- moves read as acute.")

In [ ]:
# Visualize whichever metric is trending hardest -- generic, not hardcoded to
# a specific column, so this stays useful if a different metric trends next time.
trending = trend_tbl[trend_tbl["trending"]]
if not trending.empty:
    top_metric = trending["r_vs_day"].abs().idxmax()
    s = daily_full[top_metric].dropna()
    x = np.arange(len(s))
    slope, intercept = np.polyfit(x, s.to_numpy(), 1)

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(s.index, s.to_numpy(), color="#eb6834", lw=2, marker="o", markersize=4)
    ax.plot(s.index, slope * x + intercept, color="#2a78d6", lw=1.5, ls="--")
    ax.set_ylabel(top_metric)
    ax.set_title(f"{top_metric}: trend across the window ({slope:+.4f}/day, r={trend_tbl.loc[top_metric,'r_vs_day']:+.2f})")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No metric cleared the trend threshold this period -- everything reads as day-to-day noise.")

In [ ]:
def composite_stress_index(df, baseline=None):
    """Unweighted z-score blend across whichever recovery metrics are
    present: low HRV, high resting HR, high respiratory rate, short sleep,
    a low deep-sleep share, and more time awake during the sleep session
    all push the index up (more stressed).

    Pass `baseline` (a dict of {metric: {"mean":.., "std":..}}, e.g. from
    `load_baseline()`) to score a period against a *fixed* prior reference
    instead of its own mean/std -- that's what makes scores comparable
    across months. With no baseline, 0 = this period's own average.
    Directional, not a validated clinical score, and it can't tell physical
    load apart from psychological load -- see the annotations section below
    for that."""
    def mean_std(col):
        if baseline and col in baseline:
            return baseline[col]["mean"], baseline[col]["std"]
        return df[col].mean(), df[col].std()

    z = pd.DataFrame(index=df.index)
    if "hrv" in df:
        m, s = mean_std("hrv")
        z["hrv"] = -((df["hrv"] - m) / s)
    if "rhr" in df:
        m, s = mean_std("rhr")
        z["rhr"] = (df["rhr"] - m) / s
    if "resp_rate" in df:
        m, s = mean_std("resp_rate")
        z["resp_rate"] = (df["resp_rate"] - m) / s
    if "sleep_total" in df:
        m, s = mean_std("sleep_total")
        z["sleep_total"] = -((df["sleep_total"] - m) / s)
    if "sleep_deep_pct" in df:
        m, s = mean_std("sleep_deep_pct")
        z["sleep_deep_pct"] = -((df["sleep_deep_pct"] - m) / s)
    if "sleep_awake" in df:
        m, s = mean_std("sleep_awake")
        z["sleep_awake"] = (df["sleep_awake"] - m) / s
    return z.mean(axis=1)


daily_full["stress_index"] = composite_stress_index(daily_full)
print("Most stressed:")
print(daily_full["stress_index"].sort_values(ascending=False).head(5).round(2))
print("\nMost recovered:")
print(daily_full["stress_index"].sort_values().head(5).round(2))

### Freezing this period as the baseline

`PERIOD_START`–`PERIOD_END` (Jul 5–24) is treated as the reference period going forward: its
per-metric mean/std are saved to `<SUBJECT>_stress_baseline.json`. Future months load that file
and score against these fixed numbers via `composite_stress_index(new_df, baseline=...)`, instead
of each month re-centering on itself — that's what makes "more/less stressed than usual" mean the
same thing release to release.

In [ ]:
import json

BASELINE_METRICS = ["hrv", "rhr", "resp_rate", "sleep_total", "sleep_deep_pct", "sleep_awake"]
BASELINE_PATH = DATA_DIR / f"{SUBJECT}_stress_baseline.json"


def save_baseline(df, path, metrics=BASELINE_METRICS, period=(PERIOD_START, PERIOD_END)):
    stats = {
        m: {"mean": float(df[m].mean()), "std": float(df[m].std()), "n": int(df[m].notna().sum())}
        for m in metrics
        if m in df and df[m].notna().sum() > 1
    }
    payload = {"subject": SUBJECT, "period_start": period[0], "period_end": period[1], "metrics": stats}
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return payload


def load_baseline(path):
    with open(path) as f:
        payload = json.load(f)
    return payload["metrics"]  # {metric: {"mean":.., "std":.., "n":..}}, ready for composite_stress_index(baseline=...)


baseline_payload = save_baseline(daily_full, BASELINE_PATH)
print(f"Saved baseline ({baseline_payload['period_start']} to {baseline_payload['period_end']}) -> {BASELINE_PATH}")
baseline_payload

In [ ]:
# Worked example for a future export -- uncomment once a new HealthAutoExport
# CSV (e.g. August) exists in this folder. It scores the new period against
# the frozen Jul 5-24 baseline rather than its own mean/std.
#
# next_health_csv = sorted(glob.glob(str(DATA_DIR / f"{SUBJECT}_HealthAutoExport-*.csv")))[-1]
# next_daily = drop_partial_last_day(load_daily_export(next_health_csv))
# baseline = load_baseline(BASELINE_PATH)
# next_daily["stress_index"] = composite_stress_index(next_daily, baseline=baseline)
# next_daily["stress_index"].describe()

### Scoring a single day outside the window

A day doesn't have to belong to a new export to be worth scoring against the baseline -- it might
just fall outside `PERIOD_START`-`PERIOD_END` in the *same* file, like a specific event day.
`daily_all` (built above, before trimming) keeps every day, so this works today: Jul 25 2026 was
the athlete's test day, right after the baseline window ends.

In [ ]:
test_day = daily_all.loc[["2026-07-25"]]
baseline = load_baseline(BASELINE_PATH)
scored = composite_stress_index(test_day, baseline=baseline)
print(f"Jul 25 stress index vs. Jul 5-24 baseline: {scored.iloc[0]:+.2f}")
test_day[[c for c in ["hrv", "rhr", "resp_rate", "sleep_total", "sleep_deep_pct", "sleep_awake"] if c in test_day]].round(2)

In [ ]:
by_type = (
    workouts.groupby("type")
    .agg(sessions=("type", "count"), total_min=("duration_min", "sum"), total_kcal=("active_kcal", "sum"), avg_hr=("avg_hr", "mean"))
    .round(1)
    .sort_values("total_kcal", ascending=False)
)
by_type

In [ ]:
daily_load = workouts.groupby("date").agg(kcal=("active_kcal", "sum"), types=("type", lambda s: "+".join(s)))
daily_load

In [ ]:
# Two independent measures on different scales -> two stacked charts,
# never a dual (twin) y-axis.
COLORS = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100", violet="#4a3aa7", red="#e34948")

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(daily_full.index, daily_full["hrv"], color=COLORS["blue"], lw=2)
axes[0].set_ylabel("HRV (ms)")
axes[1].plot(daily_full.index, daily_full["rhr"], color=COLORS["orange"], lw=2)
axes[1].set_ylabel("Resting HR (bpm)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.autofmt_xdate()
fig.suptitle("Recovery trend")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
colors = [COLORS["red"] if v >= 0 else COLORS["blue"] for v in daily_full["stress_index"]]
ax.bar(daily_full.index, daily_full["stress_index"], color=colors, width=0.8)
ax.axhline(0, color="#898781", lw=1)
ax.set_ylabel("Stress index (z)")
ax.set_title("Composite stress index")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
palette = [COLORS["blue"], COLORS["orange"], COLORS["aqua"], COLORS["yellow"], COLORS["violet"]]
ax.barh(by_type.index, by_type["total_kcal"], color=palette[: len(by_type)])
ax.invert_yaxis()
ax.set_xlabel("Total kcal")
ax.set_title("Workout mix")
plt.tight_layout()
plt.show()

### Notes for future exports

- **This period's baseline is frozen in `AH_stress_baseline.json`.** Jul 5–24 2026 (20 days) is
  the reference: the ramp-in week (missing sleep/resp data, sparse workouts before Jul 10) and the
  first, partially-tracked day were excluded before computing it. Score a new month against it with
  `composite_stress_index(new_df, baseline=load_baseline(BASELINE_PATH))` — see the worked example
  above — rather than letting each month re-center on itself.
- **`trend_decompose`/`trend_summary` flag metrics riding a slow within-window trend** (fitness
  adapting, growth, a gradually shifting situation) versus pure day-to-day noise — check this
  before reading too much into a correlation or a single day's z-score; a metric with a strong
  trend is not behaving like a stationary "noise around a fixed average" signal, which is what the
  z-score blend otherwise assumes.
- **Re-baselining is a deliberate choice, not automatic.** If a future period should become the new
  reference (e.g. after a training-phase change), rerun `save_baseline(new_daily_full, BASELINE_PATH)`
  explicitly — nothing overwrites the baseline file on its own.
- **VO2max, weight, blood pressure and SpO2 are often sparse or entirely blank** in a given
  export — `load_daily_export` only keeps columns that exist, but a kept column can still be
  mostly `NaN`; check `daily_full.count()` before trusting a metric's trend.
- To compare **two people or two months side by side**, load each with `load_daily_export` /
  `load_workouts` into separate frames (set `SUBJECT` per file) and concat with a `subject`/`period`
  column before grouping.
- If Health Auto Export changes a column's exact name, add the mapping to `DAILY_COLUMN_MAP` /
  `WORKOUT_COLUMN_MAP` — everything downstream keys off the short internal names, not the raw
  CSV headers.

---

## Optional: period context annotations

Everything below is **purely narrative context** — it reads from `<SUBJECT>_events.json` if that
file exists, and it never feeds into `daily_full`, the correlations, or `composite_stress_index`.
Nothing above this point depends on it. A future period with no events file, or no entries inside
`PERIOD_START`–`PERIOD_END`, just shows an empty table and skips the annotated chart — so this
section adds nothing to distract from a plain re-run on new data; it only activates when someone
has deliberately logged something worth remembering (travel, illness, exams, a schedule change).

To log an event for a period, add an entry to `<SUBJECT>_events.json`:
`{"date": "YYYY-MM-DD", "label": "short label", "note": "longer note"}`.

In [ ]:
def load_events(path, start=None, end=None):
    """Reads a period's dated context notes, if any. Returns an empty frame
    (never an error) when the file is missing or has nothing in range --
    this is display-only context, not an analysis input."""
    if not Path(path).exists():
        return pd.DataFrame(columns=["date", "label", "note"])
    events = pd.DataFrame(json.load(open(path)))
    events["date"] = pd.to_datetime(events["date"])
    if start is not None:
        events = events[events["date"] >= pd.Timestamp(start)]
    if end is not None:
        events = events[events["date"] <= pd.Timestamp(end)]
    return events.sort_values("date").reset_index(drop=True)


EVENTS_PATH = DATA_DIR / f"{SUBJECT}_events.json"
events = load_events(EVENTS_PATH, PERIOD_START, PERIOD_END)
events

In [ ]:
# A separate figure (the plain stress chart above is left untouched) --
# only drawn when there's context to show.
if not events.empty:
    fig, ax = plt.subplots(figsize=(9, 3.2))
    colors = [COLORS["red"] if v >= 0 else COLORS["blue"] for v in daily_full["stress_index"]]
    ax.bar(daily_full.index, daily_full["stress_index"], color=colors, width=0.8)
    ax.axhline(0, color="#898781", lw=1)
    for _, ev in events.iterrows():
        ax.axvline(ev["date"], color=COLORS["violet"], lw=1, ls="--", alpha=0.7)
        ax.text(ev["date"], ax.get_ylim()[1], "  " + ev["label"], rotation=90, va="top", ha="center", fontsize=8, color=COLORS["violet"])
    ax.set_ylabel("Stress index (z)")
    ax.set_title("Composite stress index, with period context")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print("No events logged for this period -- skipping the annotated view.")

---

## Extending the analysis: 2026-07-25 to 2026-08-08

Two more exports arrived: `AH_HealthAutoExport-2026-07-25-2026-08-08.csv` and
`AH_Workouts-20260725_000000-20260808_235959.csv`. Both overlap the tail of the original exports by
a day or two (2026-07-25 and 2026-07-26) -- combining prefers the *more recently generated* file's
data for any date covered twice, since a later export has had more time to fully sync. Verified:
the original file's 2026-07-26 row was a partial day (43.83 kcal vs. 912.74 kcal for the same date
in the new export), and the one duplicated workout row -- an Outdoor Walk starting
2026-07-25 14:48 -- is byte-identical between the two files, so `keep="last"` doesn't silently drop
a real difference.

The baseline itself (`AH_stress_baseline.json`, frozen from 2026-07-05 to 2026-07-24 above) is
**not** recomputed here -- the new period is scored against it, the same way the single-day example
above scores 2026-07-25, just for a full date range instead of one day.

In [ ]:
def load_daily_exports_combined(paths):
    """Loads and concatenates every matching Health Auto Export daily CSV, preferring the most
    recently generated file's data for any date covered by more than one export."""
    paths = sorted(paths, key=lambda p: Path(p).stat().st_mtime)  # oldest -> newest
    combined = pd.concat([load_daily_export(p) for p in paths])
    return combined[~combined.index.duplicated(keep="last")].sort_index()


def load_workouts_combined(paths):
    paths = sorted(paths, key=lambda p: Path(p).stat().st_mtime)  # oldest -> newest
    combined = pd.concat([load_workouts(p) for p in paths], ignore_index=True)
    combined = combined.drop_duplicates(subset=["start"], keep="last")
    return combined.sort_values("start").reset_index(drop=True)


all_health_csvs = sorted(glob.glob(str(DATA_DIR / f"{SUBJECT}_HealthAutoExport-*.csv")))
all_workout_csvs = sorted(glob.glob(str(DATA_DIR / f"{SUBJECT}_Workouts-*.csv")))
print(f"health exports: {[Path(p).name for p in all_health_csvs]}")
print(f"workout exports: {[Path(p).name for p in all_workout_csvs]}")

daily_all_combined = drop_partial_last_day(load_daily_exports_combined(all_health_csvs))
if "sleep_deep" in daily_all_combined and "sleep_total" in daily_all_combined:
    daily_all_combined["sleep_deep_pct"] = daily_all_combined["sleep_deep"] / daily_all_combined["sleep_total"]

workouts_combined = load_workouts_combined(all_workout_csvs)
print(f"combined daily range: {daily_all_combined.index.min().date()} -> {daily_all_combined.index.max().date()} ({len(daily_all_combined)} days)")

In [ ]:
NEW_PERIOD_START = "2026-07-25"
NEW_PERIOD_END = "2026-08-08"

baseline = load_baseline(BASELINE_PATH)
next_period = daily_all_combined.loc[NEW_PERIOD_START:NEW_PERIOD_END].copy()
next_period["stress_index"] = composite_stress_index(next_period, baseline=baseline)

print(f"{NEW_PERIOD_START} to {NEW_PERIOD_END} ({len(next_period)} days) vs. {PERIOD_START}-{PERIOD_END} baseline")
print("\nMost stressed:")
print(next_period["stress_index"].sort_values(ascending=False).head(5).round(2))
print("\nMost recovered:")
print(next_period["stress_index"].sort_values().head(5).round(2))
next_period[[c for c in ["hrv", "rhr", "resp_rate", "sleep_total", "sleep_deep_pct", "sleep_awake", "stress_index"] if c in next_period]].round(2)

In [ ]:
# Continuous view: baseline window's own-mean stress index (from ah10plots2 above) followed by
# the new period, both scored -- directly or via the frozen baseline -- against the same Jul 5-24
# reference, so the two halves of the bar chart are on the same scale.
combined_stress = pd.concat([daily_full["stress_index"], next_period["stress_index"]]).sort_index()

fig, ax = plt.subplots(figsize=(10, 2.8))
colors = [COLORS["red"] if v >= 0 else COLORS["blue"] for v in combined_stress]
ax.bar(combined_stress.index, combined_stress, color=colors, width=0.8)
ax.axhline(0, color="#898781", lw=1)
boundary = pd.Timestamp(PERIOD_END) + pd.Timedelta(hours=12)
ax.axvline(boundary, color=COLORS["violet"], lw=1, ls="--")
ax.text(boundary, ax.get_ylim()[1], "  baseline ends", rotation=90, va="top", ha="left", fontsize=8, color=COLORS["violet"])
ax.set_ylabel("Stress index (z)")
ax.set_title("Composite stress index -- baseline window + extended period (vs. Jul 5-24 baseline)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
new_workouts = workouts_combined[
    (workouts_combined["date"] >= NEW_PERIOD_START) & (workouts_combined["date"] <= NEW_PERIOD_END)
].reset_index(drop=True)

by_type_new = (
    new_workouts.groupby("type")
    .agg(sessions=("type", "count"), total_min=("duration_min", "sum"), total_kcal=("active_kcal", "sum"), avg_hr=("avg_hr", "mean"))
    .round(1)
    .sort_values("total_kcal", ascending=False)
)
print(f"{len(new_workouts)} workouts, {NEW_PERIOD_START} to {NEW_PERIOD_END}")
by_type_new

**Going forward:** re-run the two cells above (file discovery + combine, then scoring) whenever
another export lands -- `load_daily_exports_combined`/`load_workouts_combined` pick up every
`AH_HealthAutoExport-*.csv`/`AH_Workouts-*.csv` already in the folder automatically. Only
`NEW_PERIOD_START`/`NEW_PERIOD_END` need to change by hand, to whichever period should be scored
and plotted next.